# Try EASE-Delta

**Keep a task's decisions current as its messages change.** You say what must be true before something can be done; messages arrive, get corrected, get withdrawn. EASE-Delta reads each message against each requirement once, and after every change tells you what is ready, what is blocked and why, and which single question is worth asking. It proposes; it never acts.

This notebook runs on a free Colab machine (CPU is enough; a GPU is faster). The first run downloads the model, 1.6 GB, from the Hugging Face Hub.

[Code and documentation](https://github.com/jithinsaireddy/ease-delta) · [Measured results](https://github.com/jithinsaireddy/ease-delta/blob/main/docs/RESULTS.md) · [Model](https://huggingface.co/jithinpothireddy21/ease-delta)

In [ ]:
%pip install -q "ease-delta @ git+https://github.com/jithinsaireddy/ease-delta"

## 1. Read one claim against one passage

Does the passage support the claim, refute it, or not settle it at all? The reader was trained to say *not enough info* when a passage is about something else.

In [ ]:
from ease import EvidenceReader

reader = EvidenceReader.from_pretrained()   # jithinpothireddy21/ease-delta

claim = "The client has approved the final design."
for passage in ["Email from the client: we approve the final design, please go ahead.",
                "Email from the client: we cannot approve the design yet; the colours are wrong.",
                "The office is closed on the first Monday of the month for staff training."]:
    print(reader.read(claim, passage))

## 2. Keep a task current

Requirements are plain sentences; actions say which requirements they need (`and`, `or`, `not`, `at least 2 of (...)`). Say who is speaking in each message: the reader sees only the text. `about` says which requirement a message concerns; leave it out and the message is read against every requirement.

In [ ]:
from ease import Tracker

task = Tracker.define(
    requirements={
        "approved": "Acme has approved the final design.",
        "date": "Acme has confirmed the delivery date.",
        "nda": "The non-disclosure agreement has been signed by both parties.",
    },
    actions={"send_packet": {"requires": "approved and date and nda", "description": "Send the delivery packet"}},
    reader=reader,
)
task.add("mail-1", "Email from Acme: we approve the final design, please go ahead.", about=["approved"])
task.add("mail-2", "Email from Acme: we confirm delivery on 12 March.", about=["date"])
task.add("nda", "The NDA has been signed by Acme and countersigned by us.", about=["nda"])
print(task.status())

READY needs 90% confidence by default, so the tracker names the answer that would get there. A person's answer is read exactly. Then the client changes their mind:

In [ ]:
print(task.confirm("nda", by="Priya in Legal"))
print(task.add("mail-3", "Email from Acme: we withdraw our approval; the colours are wrong.", about=["approved"]))
print(task.explain("send_packet"))

## 3. Start from a workflow template

Templates: client hand-off, client onboarding, campaign launch, event coordination, support follow-up, software release, group trip.

When several requirements are of the same kind (one per speaker, one per participant), say which one a message concerns with `about`. Without it, a message about one person can be read as answering for another: messages on the same subject that do not settle a requirement are the reader's measured weak spot.

In [ ]:
from ease.templates import TEMPLATES
print({k: v.name for k, v in TEMPLATES.items()})

event = Tracker.from_template("event-coordination", reader=reader, speakers=["Ann Lee", "Bo Chen"], min_speakers=1)
event.add("venue", "Email from the Grand Hall: we confirm your booking for 14 November in writing.",
          about=["venue_confirmed"])
event.add("ann", "Ann Lee: happy to confirm I will give the keynote.", about=["speaker_ann_lee"])
print(event.status())
event.add("ann", "Ann Lee: I am sorry, I have to cancel; I can no longer speak at the event.",
          about=["speaker_ann_lee"])
print(event.status())          # still possible: Bo Chen may confirm
event.add("bo", "Bo Chen: yes, count me in as a speaker.", about=["speaker_bo_chen"])
print(event.status())

## 4. Only the reader, with plain `transformers`

The reader is also published as a standard sequence classifier: `jithinpothireddy21/ease-delta-reader` (395M) and `jithinpothireddy21/ease-delta-reader-base` (149M, used here because it downloads faster). Claim first, evidence second.

In [ ]:
from transformers import pipeline

clf = pipeline("text-classification", model="jithinpothireddy21/ease-delta-reader-base", top_k=None)
clf({"text": "The deposit invoice has been paid.",
     "text_pair": "Office notice: the kitchen will be closed on Friday for maintenance."}, truncation=True)

## 5. The interactive demo

This starts the same demo as the project's web page and prints a temporary public link (valid for 72 hours). Stop the cell to end it.

In [ ]:
!git clone -q https://github.com/jithinsaireddy/ease-delta
%pip install -q gradio
!EASE_SHARE=1 python ease-delta/space/app.py